In [4]:
from langchain_core.prompts import PromptTemplate
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.runnables import RunnablePassthrough
from pydantic import BaseModel, Field, ValidationError
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.documents import Document
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.exceptions import OutputParserException
import re
from typing import Any
import json
import dotenv
dotenv.load_dotenv()


class RobustPydanticOutputParser(PydanticOutputParser):
    """
    LLM 출력 또는 Python 객체를 Pydantic 모델로 안전하게 변환하는 파서.
    - 문자열: 후처리(clean ```json) 후 parse
    - list/dict: 바로 Pydantic 객체로 변환
    - JSON 형식 강제 검사 및 ValidationError 처리
    """

    def parse(self, obj: Any):
        # 이미 list/dict이면 바로 Pydantic 객체 생성
        if isinstance(obj, (list, dict)):
            try:
                return self.pydantic_object.parse_obj(obj)
            except ValidationError as e:
                raise OutputParserException(
                    f"Validation failed on Python object: {e}\nInput: {obj}")

        # 문자열이면 후처리
        if isinstance(obj, str):
            # ```json 제거 및 공백 strip
            cleaned = re.sub(r"```json|```", "", obj).strip()
            # JSON 형식인지 강제 검사
            try:
                data = json.loads(cleaned)
            except json.JSONDecodeError as e:
                raise OutputParserException(
                    f"Invalid JSON format after cleaning: {e}\nCleaned text: {cleaned}")

            # Pydantic 객체 생성
            try:
                return self.pydantic_object.parse_obj(data)
            except ValidationError as e:
                raise OutputParserException(
                    f"Pydantic validation failed: {e}\nData: {data}")

        raise OutputParserException(f"Unsupported input type: {type(obj)}")


with open('c:\\yul2ya\\llmabok\\data\\AI 에이전트 동향_short.txt', 'r', encoding='utf-8') as f:
    file = f.read()

prompt = PromptTemplate.from_template('''
다음의 context 를 읽고, 의미있는 단위로 쪼개서 반드시 format 예제와 같은 형식으로 답해주세요. 

context: {context}
format: 
[
    {{"title": "제목", "content": "내용"}},
    ...
]

모든 content는 반드시 내용이 있어야 하며, null이 되면 안됩니다.
절대로 ```json 같은 코드 블록을 사용하지 마세요.  
오직 [ ... ] JSON 배열 형식으로만 답하세요.  
다른 텍스트, 설명, 마크다운은 금지입니다.
''')


class MeaningfulChunk(BaseModel):
    title: str = Field(description="제목")
    content: str = Field(description="내용")


class MeaningfulChunkList(BaseModel):
    chunks: list[MeaningfulChunk] = Field(description="의미있는 단위로 쪼개진 콘텐츠 목록")


output_parser = RobustPydanticOutputParser(pydantic_object=MeaningfulChunkList)

llm = ChatGoogleGenerativeAI(model='gemini-2.5-flash')

# print(file)
chain = {
    'context': lambda x: file,
    # 'format': lambda x: output_parser.get_format_instructions()
} | prompt | llm  # | output_parser
response = chain.invoke({})
response

AIMessage(content='[\n    {\n        "title": "[AI브리프 스페셜] AI 에이전트 동향 2024년 12월호 서론",\n        "content": "이 문서는 SPRi AI Brief Special의 2024년 12월호로, AI 에이전트의 최신 동향을 빅테크 기업의 사례를 중심으로 다룹니다. 생성형 AI의 확산과 함께 AI 에이전트에 대한 관심이 급증하며 향후 몇 년 안에 관련 시장이 급속히 성장할 것으로 예상됩니다. 빅테크 기업들이 이 시장에 진출하여 다양한 수익 모델을 창출하고 있으나, 기술적, 사회적, 윤리적, 법적 문제도 내포하고 있습니다. AI 에이전트는 앞으로 기술 혁신과 사회 변화를 주도하는 핵심 요소로 자리 잡으며, 기업의 업무 방식과 개인의 삶에 획기적인 변화를 가져올 것으로 전망됩니다."\n    },\n    {\n        "title": "AI 에이전트의 정의 - 학술 및 연구 관점",\n        "content": "AI 에이전트에 대한 합의된 학술적 정의는 아직 부재하지만, 관련 연구에서는 AI 모델이나 알고리즘과 구별하여 상호작용과 독립적인 의사결정 능력을 강조합니다. 스튜어트 러셀과 피터 노비그는 저서 \'인공지능 – 현대적 접근법\'에서 에이전트를 환경을 인식하고 작용하는 행위자로, 합리적 에이전트를 최선의 결과를 달성하기 위해 행동하는 행위자로 정의했습니다. 알란 찬 등은 \'AI 에이전트에 대한 가시성\' 논문에서 더 큰 자율성, 외부 도구 접근, 장기적 목표 달성을 위한 안정적인 적응, 계획 및 지속적 행동 능력을 갖춘 시스템을 AI 에이전트 또는 에이전틱 시스템이라 설명합니다."\n    },\n    {\n        "title": "AI 에이전트의 정의 - 산업 및 표준 관점",\n        "content": "가트너는 AI 에이전트를 \'에이젠틱 AI\'로 칭하며, AI 기술을 사용하여 작업을 완료하고 목표를 달성하는 목표 중심 소프트웨어 엔터티로 정의합니

In [7]:
import json
json.loads(response.content)

[{'title': '[AI브리프 스페셜] AI 에이전트 동향 2024년 12월호 서론',
  'content': '이 문서는 SPRi AI Brief Special의 2024년 12월호로, AI 에이전트의 최신 동향을 빅테크 기업의 사례를 중심으로 다룹니다. 생성형 AI의 확산과 함께 AI 에이전트에 대한 관심이 급증하며 향후 몇 년 안에 관련 시장이 급속히 성장할 것으로 예상됩니다. 빅테크 기업들이 이 시장에 진출하여 다양한 수익 모델을 창출하고 있으나, 기술적, 사회적, 윤리적, 법적 문제도 내포하고 있습니다. AI 에이전트는 앞으로 기술 혁신과 사회 변화를 주도하는 핵심 요소로 자리 잡으며, 기업의 업무 방식과 개인의 삶에 획기적인 변화를 가져올 것으로 전망됩니다.'},
 {'title': 'AI 에이전트의 정의 - 학술 및 연구 관점',
  'content': "AI 에이전트에 대한 합의된 학술적 정의는 아직 부재하지만, 관련 연구에서는 AI 모델이나 알고리즘과 구별하여 상호작용과 독립적인 의사결정 능력을 강조합니다. 스튜어트 러셀과 피터 노비그는 저서 '인공지능 – 현대적 접근법'에서 에이전트를 환경을 인식하고 작용하는 행위자로, 합리적 에이전트를 최선의 결과를 달성하기 위해 행동하는 행위자로 정의했습니다. 알란 찬 등은 'AI 에이전트에 대한 가시성' 논문에서 더 큰 자율성, 외부 도구 접근, 장기적 목표 달성을 위한 안정적인 적응, 계획 및 지속적 행동 능력을 갖춘 시스템을 AI 에이전트 또는 에이전틱 시스템이라 설명합니다."},
 {'title': 'AI 에이전트의 정의 - 산업 및 표준 관점',
  'content': "가트너는 AI 에이전트를 '에이젠틱 AI'로 칭하며, AI 기술을 사용하여 작업을 완료하고 목표를 달성하는 목표 중심 소프트웨어 엔터티로 정의합니다. 에이젠틱 AI는 명시적인 입력 없이 지침을 받고, 계획을 세우며, 도구를 사용하여 작업을 완료하고 동적인 출력을 생성할 수 있습니다. ISO/IEC는 AI 관련 표준에서 에이전트

In [13]:
from ragas.testset import TestsetGenerator
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI
import dotenv
dotenv.load_dotenv()

llm = ChatGoogleGenerativeAI(model="gemini-2.0-flash")

embeddings = HuggingFaceEmbeddings(model_name="Qwen/Qwen3-Embedding-0.6B")

# pip install ragas rapidfuzz
generator = TestsetGenerator.from_langchain(llm, embeddings)

docs = [
    Document(page_content=doc['content'])
    for doc in json.loads(response.content)
]

dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

df = dataset.to_pandas()
df.to_csv('ragas/dataset.csv', index=False)

Applying SummaryExtractor:   0%|          | 0/9 [00:00<?, ?it/s]

Applying CustomNodeFilter:   0%|          | 0/9 [00:00<?, ?it/s]

Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 13
}
].
Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 4.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and bil

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/27 [00:00<?, ?it/s]

Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 5
}
].
Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and bill

...

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. [violations {
  quota_metric: "generativelanguage.googleapis.com/generate_content_free_tier_requests"
  quota_id: "GenerateRequestsPerMinutePerProjectPerModel-FreeTier"
  quota_dimensions {
    key: "model"
    value: "gemini-2.0-flash"
  }
  quota_dimensions {
    key: "location"
    value: "global"
  }
  quota_value: 15
}
, links {
  description: "Learn more about Gemini API quotas"
  url: "https://ai.google.dev/gemini-api/docs/rate-limits"
}
, retry_delay {
  seconds: 52
}
].
Retrying langchain_google_genai.chat_models._achat_with_retry.<locals>._achat_with_retry in 2.0 seconds as it raised ResourceExhausted: 429 You exceeded your current quota, please check your plan and bil

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

OSError: Cannot save file into a non-existent directory: 'ragas'

In [10]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")
vector_store = InMemoryVectorStore(embeddings)

docs = [
    Document(page_content=doc['content'])
    for doc in json.loads(response.content)
]
vector_store.add_documents(docs)
vector_store.dump('data/ai_trends_vector_store')

In [12]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

# 저장된 벡터 스토어 파일 경로
path = 'data/ai_trends_vector_store'

# 벡터 스토어 로드
vector_store = InMemoryVectorStore.load(path, embedding=embeddings)
vector_store